# NB Pruebas HU-35 a HU-39 — simulación de los casos de prueba de Jira

Este notebook ejecuta **el código real** de las celdas del notebook `NB HU35_HU36_HU37_HU38_HU39`
contra datos sintéticos, con un simulador local de Fabric (`notebookutils.fs` y `spark`), y
contrasta además las salidas que esas celdas produjeron en Fabric.

Cada caso de prueba corresponde a un caso de Jira (AGX-62 a AGX-66). Estados: **PASS**, **FAIL**
y **NO VERIFICABLE** (el notebook no contiene el elemento que el caso exige).

In [1]:
import json, os, re, sys, shutil, tempfile, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.transform import from_origin
from shapely.geometry import box, Point
import matplotlib
matplotlib.use("Agg")

sys.path.insert(0, str(Path.cwd()))
from simulador_fabric import (FakeFS, FakeSpark, instalar_notebookutils, celdas_notebook,
                              salidas_texto, ejecutar)

NB_ORIGINAL = os.environ.get("NB_ORIGINAL", "NB HU35_HU36_HU37_HU38_HU39.ipynb")
CELDAS = celdas_notebook(NB_ORIGINAL)
CRS = "EPSG:21896"
TMP = Path(tempfile.mkdtemp(prefix="agx_pruebas_"))
RESULTADOS = []

def registrar(hu, cp, caso, esperado, obtenido, estado, evidencia):
    RESULTADOS.append(dict(HU=hu, CP=cp, Caso=caso, Esperado=esperado,
                           Obtenido=obtenido, Estado=estado, Evidencia=evidencia))
    print(f"[{estado}] {cp} · {caso}\n         {obtenido}")

def escribir_tif(ruta, matriz, x0, y0, res=10, nodata=-9999):
    ruta.parent.mkdir(parents=True, exist_ok=True)
    with rasterio.open(ruta, "w", driver="GTiff", height=matriz.shape[0], width=matriz.shape[1],
                       count=1, dtype="float32", crs=CRS,
                       transform=from_origin(x0, y0, res, res), nodata=nodata) as dst:
        dst.write(matriz.astype("float32"), 1)

print("Celdas del notebook cargadas:", list(CELDAS))
print("Carpeta temporal:", TMP)

Celdas del notebook cargadas: ['HU35_ERP', 'HU35_HU36_GIS', 'HU37', 'HU38', 'HU39_P1', 'HU39_P2']
Carpeta temporal: /tmp/agx_pruebas_z63lz8ij


## HU-35 · AGX-62 — Pipeline ETL desde el ERP y el sistema de agricultura de precisión
**CP-01:** dada una ejecución del pipeline, cuando finaliza correctamente, los registros históricos de
producción y de labores quedan disponibles en el almacén analítico y la ejecución es reprocesable.

In [2]:
# --- Ingesta ERP (celda real) con un spark simulado que expone la tabla del Warehouse ---
raiz35 = TMP / "hu35"; raiz35.mkdir()
fs35 = FakeFS(raiz35); instalar_notebookutils(fs35)
maestro = pd.DataFrame({
    "HACIENDA": ["010001", "010001", "050002"], "NOMBRE_HACIENDA": ["La Esperanza"] * 2 + ["El Edén"],
    "ZONA": ["IC02", "IC02", "IC05"], "PROPRIEDAD": ["P"] * 3, "SUERTE": ["001", "002", "001"],
    "AREA_NETA": [10.5, 8.2, 20.0], "VARIEDAD": ["CC 01-1940"] * 3, "CORTE": [3, 4, 2],
    "FECHA_SIEMBRA": ["2019-02-01"] * 3, "EDAD_ULT": [12.5, 13.0, 12.1], "EDAD": [5.0, 6.0, 4.0],
    "TIPO_CULTIVO": ["Soca"] * 3, "FECHA_APLIC_MADURANTE": [None] * 3, "ZONA_AGROECOLOGICA": ["6A"] * 3,
    "FECHA_ULT_CORTE_SIEMBRA": ["2025-10-01"] * 3, "TCHM_ULT_CORTE": [9.1, 8.7, 7.9],
    "TCH_ULTC": [120.5, 110.2, 96.0], "ULT_RDTO": [11.6, 11.2, 11.0], "LLUV_ACUM": [1500, 1450, 1600],
    "NUM_RIEGOS": [3, 2, 4], "MMACUM90": [320, 300, 410], "SDAM": [8, 9, 7], "PLANTA": ["AI01"] * 3,
})
spark35 = FakeSpark(fs35, tablas={"FACT_sql_BI_MAESTRO_SUERTE": maestro})
for _ in range(2):                              # dos ejecuciones seguidas = reproceso
    ejecutar(CELDAS["HU35_ERP"], {"spark": spark35})

parquet = raiz35 / "Files/CONSOLIDADO/BIOSALC/df_crono.parquet"
leido = pd.read_parquet(parquet)
consulta = spark35.consultas[0]
produccion_ok = {"TCH_ULTC", "TCHM_ULT_CORTE", "AREA_NETA", "CORTE"} <= set(leido.columns)
reproceso_ok = [e["modo"] for e in spark35.escrituras] == ["overwrite", "overwrite"] and len(leido) == 3

# --- Consolidación GIS (celda real) ejecutada dos veces sobre un Lakehouse simulado ---
def preparar_lakehouse_gis(raiz):
    f = raiz / "Files"
    escribir_tif(f / "Data_MapasProductividad/MP_TCHM_2026.tif", np.random.rand(20, 20) * 10, 1060000, 870200)
    escribir_tif(f / "Geodata_Fertilizacion/P_TCHM.tif", np.random.rand(20, 20), 1060000, 870200)
    for v in ["CORTE", "LLUVIAS_0_-_3_MESES", "NUMRIEGOS", "VARIEDAD_TIFF"]:
        escribir_tif(f / f"RASTER_HISTORICOS/2025_{v}.tif", np.full((20, 20), 3.0), 1060000, 870200)
    escribir_tif(f / "Rosemberg_despoblacion/test_romoreno.tif", np.random.rand(20, 20), 1060000, 870200)
    (f / "RASTER_HISTORICOS/leeme.txt").write_text("no es tiff")
    capas = f / "CapasGis"; capas.mkdir(parents=True)
    for nombre in ["Ste", "Sue"]:
        gpd.GeoDataFrame({"CODSIA": ["010001001"]}, geometry=[box(1060000, 870000, 1060200, 870200)],
                         crs=CRS).to_file(capas / f"{nombre}.shp")
    gpd.GeoDataFrame({"CODSIA": ["X"]}, geometry=[box(0, 0, 1, 1)], crs="EPSG:4326").to_file(capas / "WGS.shp")

raiz_gis = TMP / "hu35_gis"; preparar_lakehouse_gis(raiz_gis)
fs_gis = FakeFS(raiz_gis); instalar_notebookutils(fs_gis)
spark_gis = FakeSpark(fs_gis)
ejecuciones = []
for i in range(2):
    ns = ejecutar(CELDAS["HU35_HU36_GIS"], {"spark": spark_gis})
    ejecuciones.append(pd.DataFrame(ns["resultados"]))
    import time; time.sleep(1.1)               # nombre de ejecución distinto (marca de tiempo)

e1, e2 = ejecuciones
copiados_1 = int((e1.estado == "COPIADO_VALIDADO").sum())
omitidos_2 = int((e2.estado == "OMITIDO_EXISTENTE").sum())
rechazado_wgs = e1.loc[e1.archivo == "WGS", "estado"].tolist()

# Evidencia de la ejecución real en Fabric (04/11/2026)
txt_fabric = salidas_texto(NB_ORIGINAL, "HU35_HU36_GIS")
m = re.search(r"Archivos y capas evaluados:\s+(\d+).*?Omitidos por existencia:\s+(\d+).*?Errores de archivos:\s+(\d+)",
              txt_fabric, re.S)
fabric_eval, fabric_omit, fabric_err = map(int, m.groups())

obtenido = (f"ERP: {len(leido)} suertes en df_crono.parquet con TCH/TCHM/área (modo overwrite en ambas corridas). "
            f"GIS: 1.ª corrida {copiados_1} archivos/capas copiados y validados; 2.ª corrida {omitidos_2} omitidos "
            f"por existencia, sin duplicar. Capa en WGS84 → {rechazado_wgs[0]}. Fabric 04/11: {fabric_eval} "
            f"evaluados, {fabric_omit} omitidos, {fabric_err} errores. Labores: tabla Labores_Realizadas del "
            "Warehouse (la consume D1); este notebook no la ingiere.")
estado = "PASS" if (produccion_ok and reproceso_ok and copiados_1 > 0 and omitidos_2 == copiados_1
                    and rechazado_wgs == ["RECHAZADO_CRS"] and fabric_err == 0) else "FAIL"
registrar("HU-35", "AGX-62-HU-35-CP-01", "Ejecución del pipeline: producción y labores disponibles y reprocesables",
          "Histórico disponible en el almacén y reproceso sin duplicar", obtenido, estado,
          "Celdas ERP y GIS ejecutadas dos veces + salida de Fabric")

#############################################
DF CRONO GUARDADO DE FORMA PERSISTENTE: 
  Ruta Lakehouse Files: Files/CONSOLIDADO/BIOSALC/df_crono.parquet
#############################################
#############################################
DF CRONO GUARDADO DE FORMA PERSISTENTE: 
  Ruta Lakehouse Files: Files/CONSOLIDADO/BIOSALC/df_crono.parquet
#############################################

#+-+--+--+-+-+--+--+-+-+--+--+-
[FASE 1/7] Configuración de rutas y fuentes completada.


#+-+--+--+-+-+--+--+-+-+--+--+-
[FASE 2/7] Carpetas de consolidación y temporales preparadas.

             Destino: Files/CONSOLIDADO
             Ejecución: 20261105T171331Z

#+-+--+--+-+-+--+--+-+-+--+--+-
[FASE 3/7] Funciones de validación TIFF y Shapefile preparadas.


#+-+--+--+-+-+--+--+-+-+--+--+-
[FASE 4/7] Consultando fuente: TCH_TCHM

             Elementos encontrados: 1
  [VALIDADO ORIGEN] MP_TCHM_2026.tif
  [COPIA TEMPORAL] MP_TCHM_2026.tif
  [VALIDACIÓN TEMPORAL] MP_TCHM_2026.tif
  [PUBLIC

  [VALIDACIÓN CRS] EPSG:21896
  [COPIA TEMPORAL] Archivos de la capa copiados


  [VALIDACIÓN TEMPORAL] Integridad y CRS verificados
  [OK] Ste | Capa completa publicada y validada en EPSG:21896

  [CAPA GIS] Sue
  [VALIDACIÓN] Componentes SHP, DBF, SHX y PRJ presentes
  [VALIDACIÓN CRS] EPSG:21896
  [COPIA TEMPORAL] Archivos de la capa copiados
  [VALIDACIÓN TEMPORAL] Integridad y CRS verificados
  [OK] Sue | Capa completa publicada y validada en EPSG:21896

  [CAPA GIS] WGS
  [VALIDACIÓN] Componentes SHP, DBF, SHX y PRJ presentes
  [RECHAZADO] WGS | CRS no permitido. Se requiere EPSG:21896; el archivo identifica EPSG:4326

#+-+--+--+-+-+--+--+-+-+--+--+-
[FASE 5/7] Fuente finalizada: 
CAPAS_GIS
             Capas identificadas: 3
             Capas copiadas: 2
             Capas omitidas: 0
             Capas rechazadas por validación: 1
             Errores: 0

#+-+--+--+-+-+--+--+-+-+--+--+-
[FASE 6/7] Registro de control guardado correctamente.

             Archivo: Files/CONSOLIDADO/CONTROL_ETL_20261105T171331Z.json


#######################################


#+-+--+--+-+-+--+--+-+-+--+--+-
[FASE 1/7] Configuración de rutas y fuentes completada.


#+-+--+--+-+-+--+--+-+-+--+--+-
[FASE 2/7] Carpetas de consolidación y temporales preparadas.

             Destino: Files/CONSOLIDADO
             Ejecución: 20261105T171332Z

#+-+--+--+-+-+--+--+-+-+--+--+-
[FASE 3/7] Funciones de validación TIFF y Shapefile preparadas.


#+-+--+--+-+-+--+--+-+-+--+--+-
[FASE 4/7] Consultando fuente: TCH_TCHM

             Elementos encontrados: 1
  [OMITIDO] MP_TCHM_2026.tif | Ya existe

#+-+--+--+-+-+--+--+-+-+--+--+-
[FASE 5/7] Fuente finalizada: 
TCH_TCHM
             TIFF encontrados: 1
             Copiados: 0
             Omitidos: 1
             Errores: 0

#+-+--+--+-+-+--+--+-+-+--+--+-
[FASE 4/7] Consultando fuente: SUELOS_QUIMICOS

             Elementos encontrados: 1
  [OMITIDO] P_TCHM.tif | Ya existe

#+-+--+--+-+-+--+--+-+-+--+--+-
[FASE 5/7] Fuente finalizada: 
SUELOS_QUIMICOS
             TIFF encontrados: 1
             Copiados: 0
          

[PASS] AGX-62-HU-35-CP-01 · Ejecución del pipeline: producción y labores disponibles y reprocesables
         ERP: 3 suertes en df_crono.parquet con TCH/TCHM/área (modo overwrite en ambas corridas). GIS: 1.ª corrida 9 archivos/capas copiados y validados; 2.ª corrida 9 omitidos por existencia, sin duplicar. Capa en WGS84 → RECHAZADO_CRS. Fabric 04/11: 196 evaluados, 196 omitidos, 0 errores. Labores: tabla Labores_Realizadas del Warehouse (la consume D1); este notebook no la ingiere.


**CP-02:** dado un archivo fuente con codificación utf-8-sig y separador punto y coma, cuando se procesa,
se ingiere sin errores de parseo.

In [3]:
lee_csv = any(re.search(r"\.csv|read_csv|format\(\"csv\"\)|sep\s*=\s*\";\"", CELDAS[k])
              for k in ("HU35_ERP", "HU35_HU36_GIS"))
registrar("HU-35", "AGX-62-HU-35-CP-02", "Archivo utf-8-sig con separador «;»",
          "Se ingiere sin errores de parseo",
          "El notebook no lee archivos CSV: el ERP se consulta por SQL (dbo.FACT_sql_BI_MAESTRO_SUERTE) y la "
          "parte GIS solo maneja TIFF y Shapefile. Solo el .prj se decodifica con utf-8-sig." if not lee_csv
          else "Se encontró lectura de CSV",
          "NO VERIFICABLE" if not lee_csv else "PASS",
          "Revisión de las celdas HU-35")

[NO VERIFICABLE] AGX-62-HU-35-CP-02 · Archivo utf-8-sig con separador «;»
         El notebook no lee archivos CSV: el ERP se consulta por SQL (dbo.FACT_sql_BI_MAESTRO_SUERTE) y la parte GIS solo maneja TIFF y Shapefile. Solo el .prj se decodifica con utf-8-sig.


## HU-36 · AGX-63 — Integrar suelos, clima, variedad y telemetría de maquinaria
**CP-01:** dada una suerte cualquiera, su ficha analítica presenta suelo, clima, variedad y labores mecanizadas.

In [4]:
nombres_fabric = re.findall(r"\[OMITIDO\] (\S+)", txt_fabric)
familias = {
    "Suelo": [n for n in nombres_fabric if n in ("Sue", "P_TCHM.tif")],
    "Clima": [n for n in nombres_fabric if "LLUVIAS" in n or "LUVIAS" in n or "NUMRIEGOS" in n],
    "Variedad": [n for n in nombres_fabric if "VARIEDAD" in n],
    "Maquinaria / labores mecanizadas": [n for n in nombres_fabric if re.search(r"TELEMETR|MAQUIN|LABOR", n, re.I)],
}
# Ficha de una suerte con la base consolidada simulada: un valor por familia en el centro del lote
centro = Point(1060100, 870100)
ficha = {}
for fam, archivo in {"Suelo": "Geodata_Fertilizacion/P_TCHM.tif",
                     "Clima": "RASTER_HISTORICOS/2025_LLUVIAS_0_-_3_MESES.tif",
                     "Variedad": "RASTER_HISTORICOS/2025_VARIEDAD_TIFF.tif"}.items():
    with rasterio.open(raiz_gis / "Files" / archivo) as r:
        ficha[fam] = round(float(next(r.sample([(centro.x, centro.y)]))[0]), 3)
conteo = {k: len(v) for k, v in familias.items()}
estado = "PASS" if all(conteo.values()) else "FAIL"
registrar("HU-36", "AGX-63-HU-36-CP-01", "Ficha analítica de una suerte",
          "Suelo, clima, variedad y labores mecanizadas",
          f"Capas consolidadas en Fabric por familia: {conteo}. Ficha simulada de 010001001: {ficha}. "
          "No hay fuente de telemetría ni de labores mecanizadas entre las 5 configuradas (FUENTES), ni serie de evaporímetro.",
          estado, "Salida de Fabric de la celda GIS + muestreo de rásteres consolidados")

[FAIL] AGX-63-HU-36-CP-01 · Ficha analítica de una suerte
         Capas consolidadas en Fabric por familia: {'Suelo': 2, 'Clima': 48, 'Variedad': 13, 'Maquinaria / labores mecanizadas': 0}. Ficha simulada de 010001001: {'Suelo': 0.249, 'Clima': 3.0, 'Variedad': 3.0}. No hay fuente de telemetría ni de labores mecanizadas entre las 5 configuradas (FUENTES), ni serie de evaporímetro.


**CP-02:** dada una indisponibilidad temporal de una fuente (API de telemetría), el resto de las fuentes se
integra sin bloquear el proceso. Se simula la caída de la fuente `Rosemberg_despoblacion`, porque la telemetría
no está configurada como fuente.

In [5]:
raiz_caida = TMP / "hu36_caida"; preparar_lakehouse_gis(raiz_caida)
fs_caida = FakeFS(raiz_caida, fuentes_caidas={"Rosemberg_despoblacion"}); instalar_notebookutils(fs_caida)
ns = ejecutar(CELDAS["HU35_HU36_GIS"], {"spark": FakeSpark(fs_caida)})
res = pd.DataFrame(ns["resultados"])
caida = res[res.estado == "ERROR_ORIGEN"].fuente.tolist()
integradas = sorted(res[res.estado == "COPIADO_VALIDADO"].fuente.unique())
control = list((raiz_caida / "Files/CONSOLIDADO").glob("CONTROL_ETL_*.json"))
estado = "PASS" if caida == ["DESPOBLACION"] and len(integradas) == 4 and control else "FAIL"
registrar("HU-36", "AGX-63-HU-36-CP-02", "Una fuente no disponible no bloquea el pipeline",
          "Las demás fuentes se integran y la falla queda registrada",
          f"Fuente caída: {caida} (ERROR_ORIGEN). Integradas igualmente: {integradas}. "
          f"Registro de control escrito: {bool(control)}.", estado,
          "Celda GIS real con falla inyectada en notebookutils.fs.ls")


#+-+--+--+-+-+--+--+-+-+--+--+-
[FASE 1/7] Configuración de rutas y fuentes completada.


#+-+--+--+-+-+--+--+-+-+--+--+-
[FASE 2/7] Carpetas de consolidación y temporales preparadas.

             Destino: Files/CONSOLIDADO
             Ejecución: 20261105T171333Z

#+-+--+--+-+-+--+--+-+-+--+--+-
[FASE 3/7] Funciones de validación TIFF y Shapefile preparadas.


#+-+--+--+-+-+--+--+-+-+--+--+-
[FASE 4/7] Consultando fuente: TCH_TCHM

             Elementos encontrados: 1
  [VALIDADO ORIGEN] MP_TCHM_2026.tif
  [COPIA TEMPORAL] MP_TCHM_2026.tif
  [VALIDACIÓN TEMPORAL] MP_TCHM_2026.tif
  [PUBLICACIÓN] MP_TCHM_2026.tif
  [OK] MP_TCHM_2026.tif | Copiado y validado 
+-+-+-+-+-

#+-+--+--+-+-+--+--+-+-+--+--+-
[FASE 5/7] Fuente finalizada: 
TCH_TCHM
             TIFF encontrados: 1
             Copiados: 1
             Omitidos: 0
             Errores: 0

#+-+--+--+-+-+--+--+-+-+--+--+-
[FASE 4/7] Consultando fuente: SUELOS_QUIMICOS

             Elementos encontrados: 1
  [VALIDADO ORIGEN] 

  [OK] Ste | Capa completa publicada y validada en EPSG:21896

  [CAPA GIS] Sue
  [VALIDACIÓN] Componentes SHP, DBF, SHX y PRJ presentes


  [VALIDACIÓN CRS] EPSG:21896
  [COPIA TEMPORAL] Archivos de la capa copiados


  [VALIDACIÓN TEMPORAL] Integridad y CRS verificados
  [OK] Sue | Capa completa publicada y validada en EPSG:21896

  [CAPA GIS] WGS
  [VALIDACIÓN] Componentes SHP, DBF, SHX y PRJ presentes
  [RECHAZADO] WGS | CRS no permitido. Se requiere EPSG:21896; el archivo identifica EPSG:4326

#+-+--+--+-+-+--+--+-+-+--+--+-
[FASE 5/7] Fuente finalizada: 
CAPAS_GIS
             Capas identificadas: 3
             Capas copiadas: 2
             Capas omitidas: 0
             Capas rechazadas por validación: 1
             Errores: 0

#+-+--+--+-+-+--+--+-+-+--+--+-
[FASE 6/7] Registro de control guardado correctamente.

             Archivo: Files/CONSOLIDADO/CONTROL_ETL_20261105T171333Z.json


#################################################################
ETL HU-38 | RESUMEN FINAL DE EJECUCIÓN
#################################################################
Archivos y capas evaluados: 10
Copiados y validados:       8
Omitidos por existencia:    0
Rechazados por validación:  1
Errores de arch

## HU-37 · AGX-64 — Rasterizar a 10 × 10 m sobre una malla fija
En Fabric la celda corre en `MODO_SIMULACION = True` (solo imprime las etapas). Aquí se ejecuta su **rama real
(local)** con un Excel y dos shapefiles sintéticos: en 2025 la suerte 001 es un solo polígono y en 2026 se divide
en 001A y 001B; además aparece la suerte 002 solo en 2026, para que la extensión cambie entre años.

In [6]:
datos37 = TMP / "hu37"; datos37.mkdir()
X0, Y0 = 1060003.7, 870007.3                    # origen deliberadamente no múltiplo de 10 m
lote_2025 = box(X0, Y0, X0 + 200, Y0 + 100)
gpd.GeoDataFrame({"CODSIA": ["010001001"]}, geometry=[lote_2025], crs=CRS).to_file(datos37 / "Ste_2025.shp")
gpd.GeoDataFrame({"CODSIA": ["010001001A", "010001001B", "010001002"]},
                 geometry=[box(X0, Y0, X0 + 100, Y0 + 100), box(X0 + 100, Y0, X0 + 200, Y0 + 100),
                           box(X0 + 300, Y0 + 50, X0 + 360, Y0 + 130)], crs=CRS).to_file(datos37 / "Ste_2026.shp")

ns_attr = ejecutar(CELDAS["HU37"].split("print(\"\\n\")")[0], {})   # solo la configuración: lista de ATRIBUTOS
ATR = ns_attr["ATRIBUTOS"]
filas = []
for cod, anio, capa, corte, edad in [("10001001", 2025, "Ste_2025", 3, 12.4),
                                     ("10001001A", 2026, "Ste_2026", 4, 11.0),
                                     ("10001001B", 2026, "Ste_2026", 4, 13.5),
                                     ("10001002", 2026, "Ste_2026", 1, 6.2)]:
    fila = {a: np.nan for a in ATR}
    fila.update({"CODSIA": cod, "year": anio, "Capa_producto_final": capa, "Corte": corte, "Edad": edad,
                 "Variedad_TIFF": 7, "Lluvias (0 - 3 Meses)": "315,5"})
    filas.append(fila)
with pd.ExcelWriter(datos37 / "Reporte_Biosal.xlsx") as w:
    pd.DataFrame(filas).to_excel(w, sheet_name="Datos_depurados_eda_quemar", index=False)

codigo37 = (CELDAS["HU37"]
            .replace("MODO_SIMULACION = True", "MODO_SIMULACION = False")
            .replace("ANOS = [2026]", "ANOS = [2025, 2026]"))
codigo37 = re.sub(r'r"C:\\Users\\[^"]+"', f'r"{datos37}"', codigo37)
ejecutar(codigo37, {})
salida37 = datos37 / "RASTER_HISTORICOS"

def leer(nombre):
    with rasterio.open(salida37 / nombre) as r:
        return r.read(1), r.transform, r.nodata, r.res, str(r.crs)

c25, t25, nd, res25, crs25 = leer("2025_CORTE.tif")
c26, t26, _, res26, _ = leer("2026_CORTE.tif")
e26, te26, _, _, _ = leer("2026_EDAD.tif")
print("Rásteres generados:", sorted(p.name for p in salida37.glob("*.tif")))

# CP-01: valor único replicado en cada píxel cuyo centro cae dentro de la suerte
def pixeles_dentro(transform, forma, geom):
    filas_, cols_ = np.indices(forma)
    xs, ys = rasterio.transform.xy(transform, filas_.ravel(), cols_.ravel())
    return np.array([geom.contains(Point(x, y)) for x, y in zip(xs, ys)]).reshape(forma)

dentro = pixeles_dentro(t25, c25.shape, lote_2025)
valores = np.unique(c25[dentro])
estado = "PASS" if dentro.sum() == 200 and valores.tolist() == [3.0] and res25 == (10.0, 10.0) else "FAIL"
registrar("HU-37", "AGX-64-HU-37-CP-01", "Variable de valor único por suerte",
          "Se rasteriza a 10 × 10 m y el valor se replica en cada píxel de la suerte",
          f"Suerte 010001001 (2 ha): {int(dentro.sum())} píxeles de {res25[0]:.0f} × {res25[1]:.0f} m, todos con CORTE = "
          f"{valores.tolist()}; fuera de los lotes = NoData ({nd}). CRS {crs25}.", estado,
          "Rama local de la celda HU-37 ejecutada con datos sintéticos")

# CP-02: con y sin variabilidad interna, la malla es la de 10 m alineada a múltiplos de 10
alineado = all(abs(v % 10) < 1e-6 for v in (t26.c, t26.f, te26.c, te26.f, t25.c, t25.f))
mismo_grid = (t26 == te26) and c26.shape == e26.shape
valores_edad = sorted(set(np.unique(e26[e26 != nd]).round(1)))
estado = "PASS" if alineado and mismo_grid and res26 == (10.0, 10.0) else "FAIL"
registrar("HU-37", "AGX-64-HU-37-CP-02", "Variable con o sin variabilidad entre suertes",
          "Respeta la malla de 10 × 10 m",
          f"CORTE y EDAD 2026 comparten transformación y forma {c26.shape}; origen ({t26.c:.0f}, {t26.f:.0f}) "
          f"múltiplo de 10 m; EDAD toma un valor por suerte {valores_edad}. «315,5» con coma decimal se convierte "
          "a número.", estado, "Rásteres 2026_CORTE.tif y 2026_EDAD.tif")

# CP-03: lote que se divide entre años → superposición píxel a píxel
# Posición de la malla 2025 dentro de la malla 2026, en píxeles
c0, f0 = (t25.c - t26.c) / 10, (t26.f - t25.f) / 10
entero = float(c0).is_integer() and float(f0).is_integer()
c0, f0 = int(c0), int(f0)
sub26 = c26[f0:f0 + c25.shape[0], c0:c0 + c25.shape[1]] if (f0 >= 0 and c0 >= 0) else None
coinciden = sub26 is not None and sub26.shape == c25.shape and ((c25 != nd) == (sub26 != nd)).all()
dx, dy = c0, f0
estado = "PASS" if entero and coinciden else "FAIL"
registrar("HU-37", "AGX-64-HU-37-CP-03", "Lote que se divide entre dos años",
          "Las capas se superponen píxel a píxel sobre la misma malla",
          f"La malla 2025 cae en la columna {dx} y la fila {dy} de la malla 2026 (desplazamiento entero); la huella de la suerte 001 (2025) "
          f"coincide píxel a píxel con 001A+001B (2026): {coinciden}. Las extensiones difieren ({c25.shape} vs "
          f"{c26.shape}) porque cada año usa los límites de sus polígonos.", estado,
          "Comparación de 2025_CORTE.tif y 2026_CORTE.tif")



GENERADOR RÁPIDO DE RASTER HISTÓRICOS

Años seleccionados: [2025, 2026]
Atributos: ['Variedad_TIFF', 'Corte', 'Edad', 'Rdto', 'TAH', 'TAHM', '%Sac.Cana', 'Num.Riegos', 'Vejez', 'Semanas mad.', 'Luvias', 'Lluvias (0 - 3 Meses)', 'Lluvias 10d Ant.Cos', 'Ton.Azucar']
Resolución: 10 x 10 metros


LECTURA DEL EXCEL
Leyendo únicamente las columnas necesarias...


Filas leídas: 4


DATOS SELECCIONADOS
Años seleccionados: [2025, 2026]
Registros a procesar: 4



################################################################################
# PROCESANDO 2025
################################################################################
Registros: 1
SHP diferentes: 1


----------------------------------------------------------------------
Procesando: Ste_2025
----------------------------------------------------------------------
/tmp/agx_pruebas_z63lz8ij/hu37/Ste_2025.shp


Polígonos SHP: 1
Polígonos relacionados: 1
Polígonos para raster: 1

Polígonos totales 2025: 1
Dimensiones: 21 x 11


RASTERIZANDO: Variedad_TIFF
✓ Guardado: /tmp/agx_pruebas_z63lz8ij/hu37/RASTER_HISTORICOS/2025_VARIEDAD_TIFF.tif


RASTERIZANDO: Corte
✓ Guardado: /tmp/agx_pruebas_z63lz8ij/hu37/RASTER_HISTORICOS/2025_CORTE.tif


RASTERIZANDO: Edad
✓ Guardado: /tmp/agx_pruebas_z63lz8ij/hu37/RASTER_HISTORICOS/2025_EDAD.tif


RASTERIZANDO: Rdto
No existen valores para rasterizar.


RASTERIZANDO: TAH
No existen valores para rasterizar.


RASTERIZANDO: TAHM
No existen valores para rasterizar.


RASTERIZANDO: %Sac.Cana
No existen valores para rasterizar.


RASTERIZANDO: Num.Riegos
No existen valores para rasterizar.


RASTERIZANDO: Vejez
No existen valores para rasterizar.


RASTERIZANDO: Semanas mad.
No existen valores para rasterizar.


RASTERIZANDO: Luvias
No existen valores para rasterizar.


RASTERIZANDO: Lluvias (0 - 3 Meses)
✓ Guardado: /tmp/agx_pruebas_z63lz8ij/hu37/RASTER_HISTORICOS/

Polígonos SHP: 3
Polígonos relacionados: 3
Polígonos para raster: 3

Polígonos totales 2026: 3
Dimensiones: 37 x 14


RASTERIZANDO: Variedad_TIFF
✓ Guardado: /tmp/agx_pruebas_z63lz8ij/hu37/RASTER_HISTORICOS/2026_VARIEDAD_TIFF.tif


RASTERIZANDO: Corte
✓ Guardado: /tmp/agx_pruebas_z63lz8ij/hu37/RASTER_HISTORICOS/2026_CORTE.tif


RASTERIZANDO: Edad
✓ Guardado: /tmp/agx_pruebas_z63lz8ij/hu37/RASTER_HISTORICOS/2026_EDAD.tif


RASTERIZANDO: Rdto
No existen valores para rasterizar.


RASTERIZANDO: TAH
No existen valores para rasterizar.


RASTERIZANDO: TAHM
No existen valores para rasterizar.


RASTERIZANDO: %Sac.Cana
No existen valores para rasterizar.


RASTERIZANDO: Num.Riegos
No existen valores para rasterizar.


RASTERIZANDO: Vejez
No existen valores para rasterizar.


RASTERIZANDO: Semanas mad.
No existen valores para rasterizar.


RASTERIZANDO: Luvias
No existen valores para rasterizar.


RASTERIZANDO: Lluvias (0 - 3 Meses)
✓ Guardado: /tmp/agx_pruebas_z63lz8ij/hu37/RASTER_HISTORICOS/

## HU-38 · AGX-65 — Correlaciones espaciales y base analítica
**CP-01:** dada una propiedad con textura espacial, su correlación con el TCH queda disponible como variable
agregada a nivel de suerte. Se ejecuta la celda real sobre un Lakehouse sintético con tres suertes.

In [7]:
from datetime import datetime
ANIO = datetime.now().year
base38 = TMP / "hu38/Files"
lotes = gpd.GeoDataFrame({"CODSIA": ["010001001", "010001002", "050002001"]},
                         geometry=[box(1060000, 870000, 1060200, 870200), box(1060200, 870000, 1060400, 870200),
                                   box(1070000, 880000, 1070100, 880100)], crs=CRS)   # el tercero queda fuera del TCHM
(base38 / "capas").mkdir(parents=True); lotes.to_file(base38 / "capas/ste.shp")
rng = np.random.default_rng(7)
tchm = 6 + np.tile(np.linspace(0, 6, 40), (20, 1)) + rng.normal(0, 0.05, (20, 40))
escribir_tif(base38 / f"CONSOLIDADO/TCH_TCHM/TCHM_{ANIO}.tif", tchm, 1060000, 870200)
escribir_tif(base38 / "CONSOLIDADO/COSECHA_HISTORICA/2025_EDAD.tif", 2 * tchm + rng.normal(0, 0.2, tchm.shape), 1060000, 870200)
escribir_tif(base38 / "CONSOLIDADO/COSECHA_HISTORICA/2025_LLUVIAS_0_-_3_MESES.tif", -3 * tchm + 900, 1060000, 870200)
corte = np.full(tchm.shape, 3.0); corte[:, 20:] = 4.0                    # valor único por suerte
escribir_tif(base38 / "CONSOLIDADO/COSECHA_HISTORICA/2025_CORTE.tif", corte, 1060000, 870200)

codigo38 = CELDAS["HU38"].replace('Path("/lakehouse/default/Files")', f'Path(r"{base38}")')
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    ns38 = ejecutar(codigo38, {})
det, mat = ns38["df_detalle"], ns38["df_matriz"]
r_edad = det[(det.variable == "Edad") & det.correlacion_pearson.notna()].correlacion_pearson
r_lluv = det[(det.variable == "Lluvias (0 - 3 Meses)") & det.correlacion_pearson.notna()].correlacion_pearson
r_corte = det[det.variable == "Corte"].correlacion_pearson
print(mat.round(3).to_string(index=False))
estado = "PASS" if (len(mat) == 2 and r_edad.min() > 0.9 and r_lluv.max() < -0.99
                    and r_corte.isna().all() and (Path(ns38["ruta_matriz"]).exists())) else "FAIL"
registrar("HU-38", "AGX-65-HU-38-CP-01", "Propiedad con textura espacial",
          "Correlación con el TCH disponible como variable por suerte",
          f"Matriz: {len(mat)} filas (una por suerte con datos) × {mat.shape[1]-1} rásteres. EDAD r = "
          f"{r_edad.round(3).tolist()}; LLUVIAS r = {r_lluv.round(3).tolist()}; CORTE (valor único) = NaN, como "
          "corresponde. Observación: pivot_table descarta las filas y columnas totalmente NaN, así que la suerte "
          "fuera del TCHM y la columna CORTE solo quedan en el CSV de detalle, no en la matriz.", estado,
          "Celda HU-38 real con rásteres sintéticos; CSV en utf-8-sig")

txt38 = salidas_texto(NB_ORIGINAL, "HU38")
lotes_f = int(re.search(r"Lotes: (\d+)", txt38).group(1)); tiff_f = int(re.search(r"TIFF comparativos: (\d+)", txt38).group(1))
reg_f = int(re.search(r"Registros detalle: (\d+)", txt38).group(1))
print(f"Fabric: {lotes_f} lotes × {tiff_f} TIFF = {lotes_f*tiff_f:,} (detalle: {reg_f:,}) → coherente: {lotes_f*tiff_f == reg_f}")

[FASE 1/7] Año de referencia: 2026
             Shapefile: /tmp/agx_pruebas_z63lz8ij/hu38/Files/capas/ste.shp
[FASE 3/7] Lotes cargados: 3
             Referencia: TCHM_2026.tif
[FASE 4/7] TIFF encontrados: 4
             TIFF por comparar: 3


[FASE 5/7] Polígonos y ventanas del TCHM preparados.
[FASE 6/7] TIFF 1/3: 2025_CORTE.tif
[FASE 6/7] TIFF 2/3: 2025_EDAD.tif


[FASE 6/7] TIFF 3/3: 2025_LLUVIAS_0_-_3_MESES.tif

#+-#+-#+-#+-#+-#+-#+-#+-#+-#+-#+-#+-#+-#+-#+-#+-#+-#+-#+-#+-#+-#+-
HU-38 | RESUMEN DE CORRELACIONES ESPACIALES
#+-#+-#+-#+-#+-#+-#+-#+-#+-#+-#+-#+-#+-#+-#+-#+-#+-#+-#+-#+-#+-#+-
Lotes: 3
TIFF comparativos: 3
Registros detalle: 9
Archivo detalle: /tmp/agx_pruebas_z63lz8ij/hu38/Files/CONSOLIDADO/CORRELACIONES_ESPACIALES/CORRELACIONES_TCHM_2026_DETALLE.csv
Archivo matriz: /tmp/agx_pruebas_z63lz8ij/hu38/Files/CONSOLIDADO/CORRELACIONES_ESPACIALES/CORRELACIONES_TCHM_2026_MATRIZ.csv
   CODSIA  2025_EDAD.tif  2025_LLUVIAS_0_-_3_MESES.tif
010001001          0.993                          -1.0
010001002          0.993                          -1.0
[PASS] AGX-65-HU-38-CP-01 · Propiedad con textura espacial
         Matriz: 2 filas (una por suerte con datos) × 2 rásteres. EDAD r = [0.993, 0.993]; LLUVIAS r = [-1.0, -1.0]; CORTE (valor único) = NaN, como corresponde. Observación: pivot_table descarta las filas y columnas totalmente NaN, así que la 

**CP-02:** dada la base analítica consolidada, cuando se publica, el diccionario de variables queda actualizado
en Confluence. (Búsqueda en el espacio AgroXpert del 05/11/2026: no hay página de diccionario de variables
ni página de HU-38.)

In [8]:
registrar("HU-38", "AGX-65-HU-38-CP-02", "Publicación del diccionario de variables",
          "Diccionario actualizado en Confluence",
          "No existe página de diccionario de variables en el espacio AgroXpert. Además, la base se publica como CSV "
          "en Lakehouse Files (CORRELACIONES_ESPACIALES), no como tabla del Fabric Data Warehouse.",
          "FAIL", "Búsqueda en Confluence (espacio AgroXpert)")

[FAIL] AGX-65-HU-38-CP-02 · Publicación del diccionario de variables
         No existe página de diccionario de variables en el espacio AgroXpert. Además, la base se publica como CSV en Lakehouse Files (CORRELACIONES_ESPACIALES), no como tabla del Fabric Data Warehouse.


## HU-39 · AGX-66 — Reporte de calidad de datos
**CP-01:** el reporte identifica los campos incompletos, los valores atípicos de TCH y las suertes con historia
insuficiente. Se ejecutan las dos celdas reales (EDA y depuración) con un Excel sintético.

In [9]:
datos39 = TMP / "hu39"; (datos39 / "salida").mkdir(parents=True)
ns_cfg = ejecutar(CELDAS["HU39_P1"].split("# 2. FUNCIONES AUXILIARES")[0], {})
VARS, TEXTO = ns_cfg["VARIABLES_EDA"], ns_cfg["COLUMNAS_TEXTO"]
rng = np.random.default_rng(3)
n = 120
df39 = pd.DataFrame({v: rng.normal(10, 1, n).round(2) for v in VARS})
df39["Variedad_TIFF"] = rng.integers(1, 5, n)
df39.loc[:9, "Num.Riegos"] = np.nan                       # campo incompleto
df39.loc[0, "TAH"] = 6914.675                             # atípico de TAH (como en Fabric)
df39.loc[1, "Edad"] = 0.1                                 # fuera de rango
for c in TEXTO:
    df39[c] = "x"
df39["CODSIA"] = [f"{i:09d}" for i in range(n)]
df39["year"] = 2025; df39.loc[:59, "year"] = 2024
with pd.ExcelWriter(datos39 / "Reporte_Biosal.xlsx") as w:
    df39.to_excel(w, sheet_name="Data_original", index=False)

ruta_xlsx = str(datos39 / "Reporte_Biosal.xlsx")
c1 = CELDAS["HU39_P1"].replace('"/lakehouse/default/Files/data_historica/Reporte_Biosal.xlsx"', repr(ruta_xlsx))
c2 = (CELDAS["HU39_P2"].replace('"/lakehouse/default/Files/data_historica/Reporte_Biosal.xlsx"', repr(ruta_xlsx))
      .replace('"/lakehouse/default/Files/CONSOLIDADO"', repr(str(datos39 / "salida"))))
import matplotlib.pyplot as plt
plt.show = lambda *a, **k: plt.close("all")
ns1 = ejecutar(c1, {}); ns2 = ejecutar(c2, {})

eda = ns1["calcular_eda"](ns1["df"])
incompleto = eda.loc[eda.Variable == "Num.Riegos", "Vacíos"].item()
atipicos = ns2["df_historial_rangos"].set_index("Variable")["Total_valores_excluidos"]
tiene_tch = any(re.fullmatch(r"TCH|TCHM", v) for v in VARS)
historia = any(re.search(r"historia|insuficiente|min_?zafras|años por suerte", CELDAS[k], re.I)
               for k in ("HU39_P1", "HU39_P2"))
auditoria = (datos39 / "salida/Auditoria_Valores_Atipicos.xlsx").exists()
estado = "PASS" if (incompleto == 10 and tiene_tch and historia) else "FAIL"
registrar("HU-39", "AGX-66-HU-39-CP-01", "Reporte de calidad sobre las fuentes integradas",
          "Campos incompletos, atípicos de TCH y suertes con historia insuficiente",
          f"Completitud: detecta los 10 vacíos de Num.Riegos. Atípicos: TAH {atipicos['TAH']} y Edad {atipicos['Edad']} "
          f"excluidos sin borrar filas; auditoría en Excel: {auditoria}. Falta: TCH y TCHM no están en VARIABLES_EDA "
          "(se evalúan TAH, TAHM y Rdto) y no se calcula la lista de suertes con historia insuficiente.",
          estado, "Celdas HU-39 Paso 1 y Paso 2 ejecutadas con Excel sintético")

txt39 = salidas_texto(NB_ORIGINAL, "HU39_P2")
print("Fabric (Paso 2):", re.search(r"Registros originales: ([\d,]+)", txt39).group(0))

Cargando Excel...
Registros cargados: 120
Columnas originales: 28

EDA - CALIDAD Y DISTRIBUCIÓN DE DATOS
             Variable Tipo  Registros  No vacíos  Vacíos  % vacíos  Únicos No numéricos      Media Mediana  Desv. est.   Mín     P25      P75       Máx
        Variedad_TIFF  Cat        120        120       0    0.0000       4                                                                             
                Corte  Num        120        120       0    0.0000     100            0  10.034583   9.985    0.942735  7.86  9.4325  10.6775     12.46
                 Edad  Num        120        120       0    0.0000     104            0  10.020917  10.035    1.371063   0.1   9.365  10.7425     12.91
                 Rdto  Num        120        120       0    0.0000     103            0  10.080583   10.11    0.955029  7.45   9.475    10.79     12.59
                  TAH  Num        120        120       0    0.0000     104            0  67.680292    10.2  630.295423  6.67    9.57  1


LÍMITES IQR Y VALORES CANDIDATOS A OUTLIERS
             Variable  Mín observado     Q1      Q3    IQR  Límite inferior IQR  Límite superior IQR  Candidatos inferiores  Candidatos superiores  Candidatos totales
                Corte           7.86 9.4325 10.6775 1.2450               7.5650              12.5450                      0                      0                   0
                 Edad           0.10 9.3650 10.7425 1.3775               7.2987              12.8088                      2                      1                   3
                 Rdto           7.45 9.4750 10.7900 1.3150               7.5025              12.7625                      1                      0                   1
                  TAH           6.67 9.5700 10.7975 1.2275               7.7288              12.6387                      1                      1                   2
                 TAHM           7.10 9.4275 10.7600 1.3325               7.4288              12.7587                    

,Variable,Tipo,Registros,No vacíos,Vacíos,% vacíos,Únicos,No numéricos,Media,Mediana,Desv. est.,Mín,P25,P75,Máx
0,Variedad_TIFF,Cat,120,120,0,0.0000,4,,,,,,,,
1,Corte,Num,120,120,0,0.0000,100,0,10.034583,9.985,0.942735,7.86,9.4325,10.6775,12.46
2,Edad,Num,120,120,0,0.0000,104,0,10.020917,10.035,1.371063,0.1,9.365,10.7425,12.91
3,Rdto,Num,120,120,0,0.0000,103,0,10.080583,10.11,0.955029,7.45,9.475,10.79,12.59
4,TAH,Num,120,120,0,0.0000,104,0,67.680292,10.2,630.295423,6.67,9.57,10.7975,6914.675
5,TAHM,Num,120,120,0,0.0000,104,0,10.044917,9.955,1.048768,7.1,9.4275,10.76,12.57
6,%Sac.Cana,Num,120,120,0,0.0000,103,0,9.858583,9.89,1.001797,7.0,9.1375,10.495,12.17
7,Num.Riegos,Num,120,110,10,8.3333,91,0,9.998727,10.1,1.004852,7.01,9.2075,10.725,12.65
8,Vejez,Num,120,120,0,0.0000,105,0,10.018167,10.07,1.010896,7.15,9.3275,10.7,13.19
9,Semanas mad.,Num,120,120,0,0.0000,103,0,10.0545,10.14,0.930909,7.84,9.3075,10.74,12.05



LÍMITES IQR INICIALES


,Variable,Q1,Q3,IQR,Límite inferior IQR,Límite superior IQR,Candidatos inferiores IQR,Candidatos superiores IQR,Candidatos totales IQR
0,Corte,9.4325,10.6775,1.2450,7.5650,12.5450,0,0,0
1,Edad,9.3650,10.7425,1.3775,7.2987,12.8088,2,1,3
2,Rdto,9.4750,10.7900,1.3150,7.5025,12.7625,1,0,1
3,TAH,9.5700,10.7975,1.2275,7.7288,12.6387,1,1,2
4,TAHM,9.4275,10.7600,1.3325,7.4288,12.7587,1,0,1
5,%Sac.Cana,9.1375,10.4950,1.3575,7.1012,12.5313,1,0,1
6,Num.Riegos,9.2075,10.7250,1.5175,6.9312,13.0013,0,0,0
7,Vejez,9.3275,10.7000,1.3725,7.2688,12.7587,1,1,2
8,Semanas mad.,9.3075,10.7400,1.4325,7.1588,12.8887,0,0,0
9,Luvias,9.3975,10.8075,1.4100,7.2825,12.9225,0,0,0



VALORES QUE SE EXCLUIRÁN POR VARIABLE


... (14 outputs left out)

,Variable,Q1_Antes,Q3_Antes,IQR_Antes,Límite inferior IQR_Antes,Límite superior IQR_Antes,Candidatos inferiores IQR_Antes,Candidatos superiores IQR_Antes,Candidatos totales IQR_Antes,Q1_Despues,Q3_Despues,IQR_Despues,Límite inferior IQR_Despues,Límite superior IQR_Despues,Candidatos inferiores IQR_Despues,Candidatos superiores IQR_Despues,Candidatos totales IQR_Despues
0,%Sac.Cana,9.1375,10.4950,1.3575,7.1012,12.5313,1,0,1,9.1375,10.4950,1.3575,7.1012,12.5313,1,0,1
1,Corte,9.4325,10.6775,1.2450,7.5650,12.5450,0,0,0,9.4325,10.6775,1.2450,7.5650,12.5450,0,0,0
2,Edad,9.3650,10.7425,1.3775,7.2987,12.8088,2,1,3,9.3750,10.7450,1.3700,7.3200,12.8000,1,1,2
3,Lluvias (0 - 3 Meses),9.3650,10.5225,1.1575,7.6288,12.2587,1,2,3,9.3650,10.5225,1.1575,7.6288,12.2587,1,2,3
4,Lluvias 10d Ant.Cos,9.4275,10.7925,1.3650,7.3800,12.8400,0,0,0,9.4275,10.7925,1.3650,7.3800,12.8400,0,0,0
5,Luvias,9.3975,10.8075,1.4100,7.2825,12.9225,0,0,0,9.3975,10.8075,1.4100,7.2825,12.9225,0,0,0
6,Num.Riegos,9.2075,10.7250,1.5175,6.9312,13.0013,0,0,0,9.2075,10.7250,1.5175,6.9312,13.0013,0,0,0
7,Rdto,9.4750,10.7900,1.3150,7.5025,12.7625,1,0,1,9.4750,10.7900,1.3150,7.5025,12.7625,1,0,1
8,Semanas mad.,9.3075,10.7400,1.4325,7.1588,12.8887,0,0,0,9.3075,10.7400,1.4325,7.1588,12.8887,0,0,0
9,TAH,9.5700,10.7975,1.2275,7.7288,12.6387,1,1,2,9.5700,10.7850,1.2150,7.7475,12.6075,1,0,1



VERIFICACIONES FINALES
Registros originales: 120
Registros finales:    120
Filas eliminadas:     0
Filas con algún valor atípico: 59
Valores individuales excluidos: 59

CORRECTO: se conservaron todas las filas y sus índices originales.
Los valores fuera de rango se sustituyeron únicamente en sus respectivas columnas.



PROCESO FINALIZADO
Archivo de auditoría generado:
/tmp/agx_pruebas_z63lz8ij/hu39/salida/Auditoria_Valores_Atipicos.xlsx

DataFrames disponibles:
df_original_rangos          -> Excel original sin modificaciones
df_limpio_rangos            -> Datos con valores atípicos como NaN
df_resumen_general_rangos   -> Resumen general
df_resumen_rangos_original  -> Conteos fuera de rango por variable
df_historial_rangos         -> Resumen de depuración por variable
df_auditoria_atipicos       -> Detalle de cada valor excluido
df_resumen_anios_rangos     -> Comparación de filas por año
eda_comparativo_rangos      -> Comparación del EDA
iqr_comparativo_rangos      -> Comparación de límites IQR


[FAIL] AGX-66-HU-39-CP-01 · Reporte de calidad sobre las fuentes integradas
         Completitud: detecta los 10 vacíos de Num.Riegos. Atípicos: TAH 1 y Edad 1 excluidos sin borrar filas; auditoría en Excel: True. Falta: TCH y TCHM no están en VARIABLES_EDA (se evalúan TAH, TAHM y Rdto) y no se calcula la lista de suertes con historia insuficiente.
Fabric (Paso 2): Registros originales: 60,693


**CP-02:** dadas las suertes con historia insuficiente, cuando se inicia el entrenamiento del modelo, quedan
excluidas del conjunto.

In [10]:
registrar("HU-39", "AGX-66-HU-39-CP-02", "Exclusión de suertes con historia insuficiente al entrenar",
          "Quedan excluidas del conjunto de entrenamiento",
          "El entrenamiento del modelo pertenece al Release 2 y la lista de suertes con historia insuficiente aún no "
          "se genera (ver CP-01).", "NO VERIFICABLE", "Alcance: Release 2")

[NO VERIFICABLE] AGX-66-HU-39-CP-02 · Exclusión de suertes con historia insuficiente al entrenar
         El entrenamiento del modelo pertenece al Release 2 y la lista de suertes con historia insuficiente aún no se genera (ver CP-01).


## Resumen

In [11]:
resumen = pd.DataFrame(RESULTADOS)
pd.set_option("display.max_colwidth", 120)
print(resumen[["CP", "Estado"]].to_string(index=False))
print("\nTotales:", resumen.Estado.value_counts().to_dict())
resumen.to_csv("resultados_pruebas_hu35_hu39.csv", index=False, sep=";", encoding="utf-8-sig")
shutil.rmtree(TMP, ignore_errors=True)

                CP         Estado
AGX-62-HU-35-CP-01           PASS
AGX-62-HU-35-CP-02 NO VERIFICABLE
AGX-63-HU-36-CP-01           FAIL
AGX-63-HU-36-CP-02           PASS
AGX-64-HU-37-CP-01           PASS
AGX-64-HU-37-CP-02           PASS
AGX-64-HU-37-CP-03           PASS
AGX-65-HU-38-CP-01           PASS
AGX-65-HU-38-CP-02           FAIL
AGX-66-HU-39-CP-01           FAIL
AGX-66-HU-39-CP-02 NO VERIFICABLE

Totales: {'PASS': 6, 'FAIL': 3, 'NO VERIFICABLE': 2}
